# prime-rl Offline Package Prep (uv-export + pip-download hybrid)

prime-rl uses `[tool.uv.sources]` to point several deps at git/URL/custom-index sources
(`dion`, `torchtitan`, `verifiers`, `transformers` dev, `flash-linear-attention`,
`pydantic-config`, plus the `primeintellect` PyPI mirror for env packages).

Plain `pip download prime-rl/` fails on `dion` because pip cannot read `[tool.uv.sources]`.
Workaround: use `uv export` to convert the locked deps into a requirements.txt where
git/URL sources are spelled as `pkg @ git+https://...` PEP 508 references that pip
DOES understand. Then `pip download -r reqs.txt` handles the rest.

Output layout:
```
output/
├── wheels/                       # prime-rl + every dep wheel (incl torch cu128, git pkgs)
├── prime_rl_src/                 # prime-rl source
├── models/Qwen3-0.6B/
├── datasets/reverse-text/
└── manifest.json
```

## 1. Sanity check (uv + python3.12 already on Kaggle)

In [ ]:
import os, subprocess, shutil
os.environ["WANDB_MODE"] = "disabled"
subprocess.run("which python3.12 && python3.12 --version", shell=True, check=True)
subprocess.run("which uv && uv --version", shell=True, check=True)

## 2. Clone prime-rl

In [ ]:
import subprocess
subprocess.run("rm -rf /kaggle/working/prime-rl", shell=True, check=True)
subprocess.run(
    "git clone --depth 1 https://github.com/PrimeIntellect-ai/prime-rl.git /kaggle/working/prime-rl",
    shell=True, check=True,
)
subprocess.run("ls /kaggle/working/prime-rl/uv.lock && wc -l /kaggle/working/prime-rl/uv.lock", shell=True)

## 3. Export locked requirements as a pip-compatible reqs.txt

`uv export --no-hashes --format requirements-txt` emits each package as either
`pkg==version` (PyPI) or `pkg @ git+https://...@rev` / `pkg @ https://.../wheel.whl`
(direct references). pip understands all of these natively.

In [ ]:
import subprocess
subprocess.run(
    "cd /kaggle/working/prime-rl && "
    "uv export --no-hashes --no-emit-project --format requirements-txt "
    "> /kaggle/working/reqs.txt",
    shell=True, check=True,
)
subprocess.run("wc -l /kaggle/working/reqs.txt", shell=True)
print("--- git/URL references in reqs.txt (sanity) ---")
subprocess.run("grep -E ' @ (git\\+|https://)' /kaggle/working/reqs.txt | head -20", shell=True)
print()
print("--- first 30 lines ---")
subprocess.run("head -30 /kaggle/working/reqs.txt", shell=True)

## 4. pip download all deps (PyPI + git + cu128 torch)

In [ ]:
import os, subprocess, glob

OUT_WHEELS = "/kaggle/working/output/wheels"
os.makedirs(OUT_WHEELS, exist_ok=True)

# Pass 1: download every pinned package from reqs.txt (resolves git, URL, PyPI all)
subprocess.run(
    f"python3.12 -m pip download "
    f"-r /kaggle/working/reqs.txt "
    f"--dest {OUT_WHEELS} "
    f"--no-deps "
    f"--prefer-binary "
    f"--extra-index-url https://download.pytorch.org/whl/cu128 "
    f"--extra-index-url https://hub.primeintellect.ai/primeintellect/simple/",
    shell=True, check=True,
)

# Pass 2: force cu128 torch family
subprocess.run(
    f"python3.12 -m pip download torch torchvision torchaudio "
    f"--index-url https://download.pytorch.org/whl/cu128 "
    f"--no-deps "
    f"--dest {OUT_WHEELS}",
    shell=True, check=True,
)

# Pass 3: flash-attn pre-built wheel (cu128 + torch 2.10 + cp312)
FA_URL = (
    "https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/"
    "download/v0.7.16/flash_attn-2.8.3+cu128torch2.10-cp312-cp312-linux_x86_64.whl"
)
FA_NAME = FA_URL.rsplit("/", 1)[1]
subprocess.run(f"wget -q -O {OUT_WHEELS}/{FA_NAME} {FA_URL}", shell=True, check=True)

# Pass 4: env packages from primeintellect index. pip download via reqs.txt
# silently skipped these (the index appears not to be honored by pip's resolver
# the way uv handles it). Pull each wheel by its direct lock URL instead.
PI_ENVS = {
    "reverse_text-0.1.5-py2.py3-none-any.whl":
        "https://hub.primeintellect.ai/primeintellect/reverse-text/@e3cd3fe4/reverse_text-0.1.5-py2.py3-none-any.whl",
    # add more env wheels here when needed (aime2024, alphabet_sort, etc.)
}
for name, url in PI_ENVS.items():
    print(f"  fetching {name}")
    subprocess.run(f"wget -q -O {OUT_WHEELS}/{name} {url}", shell=True, check=True)

# Drop any non-cu128 torch wheels collected in pass 1
removed = []
for prefix in ("torch-", "torchvision-", "torchaudio-"):
    for w in glob.glob(f"{OUT_WHEELS}/{prefix}*.whl"):
        if "+cu128" not in os.path.basename(w):
            os.remove(w)
            removed.append(os.path.basename(w))
print(f"Removed non-cu128 torch wheels: {removed}")

# Convert sdists to wheels
sdists = sorted(glob.glob(f"{OUT_WHEELS}/*.tar.gz") + glob.glob(f"{OUT_WHEELS}/*.zip"))
print(f"\nsdists to build into wheels: {len(sdists)}")
for sdist in sdists:
    print(f"  building {os.path.basename(sdist)}")
    subprocess.run(
        f"python3.12 -m pip wheel {sdist} --no-deps -w {OUT_WHEELS}",
        shell=True, check=True,
    )
    os.remove(sdist)

# Build prime-rl itself
subprocess.run(
    f"python3.12 -m pip wheel /kaggle/working/prime-rl --no-deps -w {OUT_WHEELS}",
    shell=True, check=True,
)

wheel_count = int(subprocess.check_output(f"ls {OUT_WHEELS} | wc -l", shell=True).decode().strip())
print(f"\nwheels collected: {wheel_count}")
subprocess.run(f"du -sh {OUT_WHEELS}", shell=True)
print()
print("--- torch + flash-attn + env wheels present ---")
subprocess.run(f"ls {OUT_WHEELS} | grep -E '^(torch|flash_attn|reverse_text|aime|alphabet)' | sort", shell=True)
print()
print("--- any sdist remaining? (should be empty) ---")
subprocess.run(f"ls {OUT_WHEELS} | grep -E '\\.(zip|tar\\.gz)$' || echo NONE", shell=True)

if wheel_count < 50:
    raise RuntimeError(f"Only {wheel_count} wheels collected -- pip download failed")

## 5. Snapshot HF model + dataset

In [ ]:
import subprocess
subprocess.run("python3.12 -m pip install -q huggingface_hub datasets", shell=True, check=True)

# Bundle every HF asset we need:
#   * Qwen3-0.6B model + reverse-text SFT data (S2, S3)
#   * Reverse-Text-RL dataset for the verifiers env (S3, sed-patched at install)
#   * LFM2.5-VL-450M for S4 (and eventual SatelliteAgent integration)
snap = '''
import os
os.environ["HF_HOME"] = "/kaggle/working/output/hf_cache"
os.environ["HF_HUB_CACHE"] = "/kaggle/working/output/hf_cache"
os.environ["HF_DATASETS_CACHE"] = "/kaggle/working/output/hf_cache/datasets"

from huggingface_hub import snapshot_download
from datasets import load_dataset

snapshot_download(
    "PrimeIntellect/Qwen3-0.6B",
    local_dir="/kaggle/working/output/models/Qwen3-0.6B",
)
snapshot_download(
    "willcb/R1-reverse-wikipedia-paragraphs-v1-1000",
    repo_type="dataset",
    local_dir="/kaggle/working/output/datasets/reverse-text",
)
snapshot_download(
    "PrimeIntellect/Reverse-Text-RL",
    repo_type="dataset",
    local_dir="/kaggle/working/output/datasets/Reverse-Text-RL",
)
snapshot_download("PrimeIntellect/Reverse-Text-RL", repo_type="dataset")
ds = load_dataset("PrimeIntellect/Reverse-Text-RL", "default", split="train")
print(f"materialized Reverse-Text-RL: {len(ds)} examples, columns={ds.column_names}")

# LFM2.5-VL for S4 (vision-language model load test).
snapshot_download(
    "LiquidAI/LFM2.5-VL-450M",
    local_dir="/kaggle/working/output/models/LFM2.5-VL-450M",
)
print("snapshots done")
'''
with open("/tmp/snap.py", "w") as f:
    f.write(snap)
subprocess.run("python3.12 /tmp/snap.py", shell=True, check=True)

print("\n--- datasets/Reverse-Text-RL contents ---")
subprocess.run("ls -la /kaggle/working/output/datasets/Reverse-Text-RL", shell=True)
print("\n--- models/LFM2.5-VL-450M contents ---")
subprocess.run("ls -la /kaggle/working/output/models/LFM2.5-VL-450M", shell=True)

## 6. Collect prime-rl source + manifest

In [ ]:
import os, subprocess, json

OUT = "/kaggle/working/output"
os.makedirs(f"{OUT}/prime_rl_src", exist_ok=True)
subprocess.run(
    f"rsync -a --exclude='.venv' --exclude='.git' --exclude='__pycache__' "
    f"/kaggle/working/prime-rl/ {OUT}/prime_rl_src/",
    shell=True, check=True,
)

print("=== Tree ===")
subprocess.run(f"ls -la {OUT}", shell=True)
print()
print("=== Sizes ===")
for sub in ["wheels", "prime_rl_src", "models", "datasets", "hf_cache"]:
    p = os.path.join(OUT, sub)
    if os.path.exists(p):
        out = subprocess.check_output(f"du -sh {p}", shell=True).decode().split()[0]
        print(f"  {sub:20s} {out}")
total = subprocess.check_output(f"du -sh {OUT}", shell=True).decode().split()[0]
print(f"\n=== Total: {total} ===")
print()
wheels = sorted(os.listdir(f"{OUT}/wheels"))
print(f"=== Wheel count: {len(wheels)} ===")
for w in wheels[:50]:
    print(f"  {w}")
if len(wheels) > 50:
    print(f"  ... and {len(wheels) - 50} more")
print()
print("=== Heavy wheels (> 50MB) ===")
for w in wheels:
    p = f"{OUT}/wheels/{w}"
    sz = os.path.getsize(p)
    if sz > 50 * 1024 * 1024:
        print(f"  {w:60s} {sz/1024/1024:.0f} MB")
print()
print("=== HF cache contents (Reverse-Text-RL) ===")
subprocess.run(f"find {OUT}/hf_cache -maxdepth 3 -type d 2>/dev/null", shell=True)
print()
print("=== Examples present? ===")
for f in ["examples/reverse_text/sft.toml", "examples/reverse_text/rl.toml"]:
    p = f"{OUT}/prime_rl_src/{f}"
    print(f"  {'OK ' if os.path.exists(p) else 'MISS '} {f}")
manifest = {"wheels_count": len(wheels), "wheels": wheels, "sizes": {}}
for sub in ["wheels", "prime_rl_src", "models", "datasets", "hf_cache"]:
    p = os.path.join(OUT, sub)
    if os.path.exists(p):
        manifest["sizes"][sub] = subprocess.check_output(f"du -sb {p}", shell=True).decode().split()[0]
with open(f"{OUT}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2)
print("\nWrote manifest.json")